# Task 4 - Open-Set Recognition

CIFAR-10 known classes and CIFAR-100 test-only unknown evaluation.

**Execution policy:** this notebook is intentionally delivered unexecuted. Set the configuration paths and switches, then run top-to-bottom when you are ready to conduct the experiment. It does not answer the report questions.

## 1. Configuration and fixed known/unknown protocol

Only CIFAR-10 training data are used to optimize models and set thresholds. CIFAR-100 unknowns are loaded only in final evaluation.

The next cell fixes CIFAR-10 training settings, the five placeholder hyperparameters, and the fixed near/far CIFAR-100 class groups. It produces no scores.

In Colab, this configuration mounts Drive. Large inputs and checkpoints live under `ATML_PA1/` in Drive; small final outputs go to this task's repository `results/` directory. The repository name is detected automatically.

In [ ]:
# Standard library and experiment dependencies.
# Install dependencies yourself before executing: torch torchvision open_clip_torch
# scikit-learn pandas matplotlib seaborn pillow scipy tqdm
import json, random, math, copy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode

SEED = 6304
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Repository and Drive paths

The next cell locates the clone, mounts Drive in Colab, and creates persistent data/checkpoint/artifact directories plus the task's small-results directory.

In [ ]:
def find_project_root():
    """Find the cloned checkout without assuming its directory name."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "requirements.txt").is_file() and (
            candidate / "task1" / "task1.ipynb"
        ).is_file():
            return candidate
    # Colab often starts in /content even when the notebook lives in a cloned repo.
    content_root = Path("/content")
    if content_root.is_dir():
        matches = [
            child
            for child in content_root.iterdir()
            if child.is_dir()
            and (child / "requirements.txt").is_file()
            and (child / "task1" / "task1.ipynb").is_file()
        ]
        if len(matches) == 1:
            return matches[0]
    raise FileNotFoundError(
        "Change into the cloned repository (or a task folder); exactly one PA1 clone must be discoverable under /content."
    )


# The clone holds code and small final outputs; Drive retains large inputs and models.
REPO_ROOT = find_project_root()
TASK_ROOT = REPO_ROOT / "task4"
REPO_RESULTS_DIR = TASK_ROOT / "results"

import os

if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive

    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ATML_PA1")
DATA_ROOT = DRIVE_ROOT / "data"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "task4"
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "task4"
EXTERNAL_DIR = DRIVE_ROOT / "external"
TORCH_CACHE_DIR = EXTERNAL_DIR / "torch_cache"
HF_CACHE_DIR = EXTERNAL_DIR / "huggingface_cache"

for directory in (
    REPO_RESULTS_DIR,
    DATA_ROOT,
    CHECKPOINT_DIR,
    ARTIFACT_DIR,
    EXTERNAL_DIR,
    TORCH_CACHE_DIR,
    HF_CACHE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

# Pretrained-weight downloads also survive a Colab runtime reset.
os.environ["TORCH_HOME"] = str(TORCH_CACHE_DIR)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)


### Reproducibility and display helpers

The next cell defines the fixed-seed behavior and small output helpers. It does not run an experiment.

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(value, f, indent=2)


def show_table(rows, title=None):
    frame = pd.DataFrame(rows)
    if title:
        print(title)
    display(frame)
    return frame


set_seed()


### Task settings

The next cell records this task's data path, fixed experiment settings, and plot/print_metrics switches. It produces no metrics.

In [ ]:
from torchvision.models import resnet18
from torchvision.transforms import RandAugment

CONFIG = {
    "data_root": DATA_ROOT,
    "download_data": False,
    "results_dir": REPO_RESULTS_DIR,
    "epochs": 100,
    "proser_epochs": 50,
    "batch_size": 128,
    "lr": 0.1,
    "proser_lr": 1e-3,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "dummy_classes": 5,
    "beta": 1.0,
    "gamma": 0.1,
    "plot": True,
    "print_metrics": True,
}
RESULTS = Path(CONFIG["results_dir"])
RESULTS.mkdir(parents=True, exist_ok=True)
C10 = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
NEAR = ["bus", "pickup_truck", "motorcycle", "tractor", "wolf", "fox", "leopard", "camel"]
FAR = ["bottle", "bowl", "chair", "clock", "keyboard", "mushroom", "sunflower", "wardrobe"]


## 2. CIFAR-10 split, CIFAR ResNet-18, and loaders

The next cell builds and saves the deterministic 90/10 CIFAR-10 split, defines CIFAR-appropriate ResNet-18 changes, and creates loaders.

In [ ]:
train_tf = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)),
    ]
)
strong_tf = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)),
    ]
)
eval_tf = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))]
)
# Check only known-class data here; CIFAR-100 remains untouched until final evaluation.
if not CONFIG["download_data"] and not (DATA_ROOT / "cifar-10-batches-py").is_dir():
    raise FileNotFoundError(
        f"CIFAR-10 is missing from Google Drive: {DATA_ROOT / 'cifar-10-batches-py'}"
    )
raw_train = datasets.CIFAR10(CONFIG["data_root"], train=True, download=CONFIG["download_data"])
raw_test = datasets.CIFAR10(CONFIG["data_root"], train=False, download=CONFIG["download_data"])
y = np.array(raw_train.targets)
rng = np.random.default_rng(SEED)
train_ids = []
val_ids = []
for c in range(10):
    ids = np.flatnonzero(y == c)
    rng.shuffle(ids)
    cut = int(0.9 * len(ids))
    train_ids += list(ids[:cut])
    val_ids += list(ids[cut:])
save_json(
    {"seed": SEED, "train_indices": train_ids, "val_indices": val_ids},
    RESULTS / "cifar10_split.json",
)


### CIFAR model and data loaders

The known-class split above is saved for reproducibility. The following code defines the unchanged CIFAR ResNet-18 and train/validation/test loaders.

In [ ]:
# This changes ImageNet ResNet-18 for 32x32 inputs: a 3x3 stride-1 stem and no max-pooling.
def cifar_resnet(outputs=10):
    m = resnet18(weights=None, num_classes=outputs)
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    return m.to(DEVICE)


class CIFARWithTransform(Dataset):
    """Apply a chosen transform without mutating torchvision's shared raw dataset."""

    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, label = self.dataset[self.indices[position]]
        return self.transform(image), label


def loaders(training_transform):
    # Validation and test never use augmentation; only the training view changes for GCSC.
    train_set = CIFARWithTransform(raw_train, train_ids, training_transform)
    val_set = CIFARWithTransform(raw_train, val_ids, eval_tf)
    test_set = CIFARWithTransform(raw_test, range(len(raw_test)), eval_tf)
    return (
        DataLoader(train_set, batch_size=CONFIG["batch_size"], shuffle=True),
        DataLoader(val_set, batch_size=256),
        DataLoader(test_set, batch_size=256),
    )


def feature_logits(m, x):
    h = m.relu(m.bn1(m.conv1(x)))
    h = m.layer1(h)
    h = m.layer2(h)
    h = m.layer3(h)
    h = m.layer4(h)
    f = torch.flatten(m.avgpool(h), 1)
    return f, m.fc(f)


## 3. Vanilla and GCSC training (validation-only checkpoint selection)

The next cell trains the Vanilla and RandAugment GCSC models under the same optimizer, schedule, seed, and validation-accuracy checkpoint rule. It saves selected checkpoints.

In [ ]:
# The only difference between Vanilla and GCSC is the specified RandAugment transform.
def train_closed_set(name, tf):
    set_seed()
    m = cifar_resnet()
    opt = torch.optim.SGD(
        m.parameters(),
        lr=CONFIG["lr"],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CONFIG["epochs"])
    train_loader, val_loader, _ = loaders(tf)
    best = -1
    state = None
    history = []
    for epoch in range(CONFIG["epochs"]):
        m.train()
        losses = []
        for x, yb in train_loader:
            _, z = feature_logits(m, x.to(DEVICE))
            loss = F.cross_entropy(z, yb.to(DEVICE))
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        scheduler.step()
        m.eval()
        ys = []
        ps = []
        with torch.no_grad():
            for x, yb in val_loader:
                _, z = feature_logits(m, x.to(DEVICE))
                ps.extend(z.argmax(1).cpu())
                ys.extend(yb)
        acc = accuracy_score(ys, ps)
        history.append({"epoch": epoch + 1, "loss": np.mean(losses), "val_accuracy": acc})
        if acc > best:
            best, state = acc, copy.deepcopy(m.state_dict())
    m.load_state_dict(state)
    torch.save({"model": m.state_dict()}, CHECKPOINT_DIR / f"{name}.pt")
    return m, history


### Fit Vanilla and GCSC

The following cell trains both known-class models in the existing order and stores their validation-selected checkpoints in Drive.

In [ ]:
vanilla, vanilla_history = train_closed_set("vanilla", train_tf)
gcsc, gcsc_history = train_closed_set("gcsc", strong_tf)


## 4. Scores from one frozen model output cache

The next cell defines a single feature/logit extraction route and the MSP, MLS, Energy, and diagonal Mahalanobis unknownness scores. These scores are evaluated on identical cached outputs.

In [ ]:
# Score functions use features and logits from the same evaluation views.
# All values are oriented so larger means more likely to be unknown.
def unknownness(cache, means=None, var=None):
    z = cache["logits"]
    p = z.softmax(1)
    max_probability = p.max(1).values
    max_logit = z.max(1).values
    logsumexp = torch.logsumexp(z, 1)
    out = {
        "msp": 1 - max_probability,
        "mls": -max_logit,
        "energy": -logsumexp,
    }
    if means is not None:
        # Broadcast [N, 512] features against ten [512]-D class means.
        feature_difference = cache["features"][:, None, :] - means[None, :, :]
        scaled_squared_difference = feature_difference.square() / var[None, None, :]
        distance_to_each_class = scaled_squared_difference.sum(2)
        out["mahalanobis"] = distance_to_each_class.min(1).values
    return out


# Estimate means and one diagonal covariance only from unaugmented known training features.
def class_statistics(train_cache):
    class_means = []
    for class_index in range(10):
        class_features = train_cache["features"][
            train_cache["labels"] == class_index
        ]
        class_means.append(class_features.mean(0))
    means = torch.stack(class_means)

    centered_groups = []
    for class_index in range(10):
        class_features = train_cache["features"][
            train_cache["labels"] == class_index
        ]
        centered_groups.append(class_features - means[class_index])
    centered = torch.cat(centered_groups)
    return means, centered.var(0, unbiased=False) + 1e-6


## 5. PROSER: classifier and data placeholders

The loss below follows Zhou et al. (2021); cite the paper and reference implementation in the README. It never loads CIFAR-100 during training.

The next cell initializes PROSER from Vanilla, adds five dummy outputs, and defines classifier-placeholder and manifold-mixup data-placeholder training. It uses CIFAR-10 only.

In [ ]:
# The final layer has 10 known logits followed by five learned dummy-placeholder logits.
class PROSER(nn.Module):
    def __init__(self, vanilla_model):
        super().__init__()
        # Save Vanilla's selected known-class layer before enlarging it with dummy outputs.
        known_classifier = copy.deepcopy(vanilla_model.fc)
        self.backbone = vanilla_model
        self.backbone.fc = nn.Linear(512, 10 + CONFIG["dummy_classes"])
        with torch.no_grad():
            self.backbone.fc.weight[:10].copy_(known_classifier.weight)
            self.backbone.fc.bias[:10].copy_(known_classifier.bias)

    def pre(self, x):
        m = self.backbone
        h = m.relu(m.bn1(m.conv1(x)))
        h = m.layer1(h)
        return m.layer2(h)

    def post(self, h):
        m = self.backbone
        h = m.layer3(h)
        h = m.layer4(h)
        return m.fc(torch.flatten(m.avgpool(h), 1))


def proser_loss(logits, yb):
    dummy_max = logits[:, 10:].max(dim=1).values.unsqueeze(1)
    augmented = torch.cat([logits[:, :10], dummy_max], dim=1)
    # Paper Eq. 5: true known class wins among known and strongest dummy.
    classification = F.cross_entropy(augmented, yb)
    # With the true class removed, the strongest dummy becomes the target.
    masked = augmented.clone()
    masked[torch.arange(len(yb), device=yb.device), yb] = -float("inf")
    dummy_targets = torch.full_like(yb, 10)
    placeholder = F.cross_entropy(masked, dummy_targets)
    return classification + CONFIG["beta"] * placeholder


# Mixing different labels prevents a synthetic placeholder from collapsing onto a known class manifold.
def different_class_pairs(yb):
    # Select a different-label partner for every feature; reject same-class pairs.
    partners = []
    for label in yb:
        candidates = torch.where(yb != label)[0]
        if len(candidates) == 0:
            raise ValueError("A placeholder batch must contain at least two known classes.")
        partner_index = torch.randint(len(candidates), (1,), device=yb.device)
        partners.append(candidates[partner_index].item())
    return torch.tensor(partners, device=yb.device)


def train_proser(vanilla_model):
    set_seed()
    m = PROSER(copy.deepcopy(vanilla_model)).to(DEVICE)
    opt = torch.optim.SGD(
        m.parameters(),
        lr=CONFIG["proser_lr"],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"],
    )
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CONFIG["proser_epochs"])
    loader, valid, _ = loaders(train_tf)
    best = -1
    state = None
    for epoch in range(CONFIG["proser_epochs"]):
        m.train()
        for x, yb in loader:
            x, yb = x.to(DEVICE), yb.to(DEVICE)
            half = len(yb) // 2
            ordinary = m.backbone(x[:half])
            cp = proser_loss(ordinary, yb[:half])
            h = m.pre(x[half:])
            pair = different_class_pairs(yb[half:])
            lam = torch.distributions.Beta(2, 2).sample((len(h),)).to(DEVICE).view(-1, 1, 1, 1)
            mixed = lam * h + (1 - lam) * h[pair]
            mixed_logits = m.post(mixed)
            # Any dummy placeholder can explain a mixed feature, so maximize the best dummy probability.
            dummy_max = mixed_logits[:, 10:].max(dim=1).values.unsqueeze(1)
            augmented_mix = torch.cat([mixed_logits[:, :10], dummy_max], dim=1)
            dummy_targets = torch.full((len(h),), 10, dtype=torch.long, device=DEVICE)
            dp = F.cross_entropy(augmented_mix, dummy_targets)
            loss = cp + CONFIG["gamma"] * dp
            opt.zero_grad()
            loss.backward()
            opt.step()
        sch.step()
        m.eval()
        ys = []
        ps = []
        with torch.no_grad():
            for x, yb in valid:
                ps.extend(m.backbone(x.to(DEVICE))[:, :10].argmax(1).cpu())
                ys.extend(yb)
        if accuracy_score(ys, ps) > best:
            best, state = accuracy_score(ys, ps), copy.deepcopy(m.state_dict())
    m.load_state_dict(state)
    torch.save({"model": m.state_dict()}, CHECKPOINT_DIR / "proser.pt")
    return m


### Train PROSER from selected Vanilla weights

This cell trains the existing placeholder model from the validation-selected Vanilla checkpoint and writes its checkpoint to Drive.

In [ ]:
proser = train_proser(vanilla)


## 6. Final unknown evaluation, validation-calibrated thresholds, ROC/distributions, and failures

Load CIFAR-100 only here, after all checkpoints and score definitions are fixed.

The next cell is the only CIFAR-100 access. It defines AUROC and validation-calibrated rejection metrics and specifies the required score plots and saved failure records.

In [ ]:
# This is the first and only CIFAR-100 access. It happens after all model training choices are fixed.
if not CONFIG["download_data"] and not (DATA_ROOT / "cifar-100-python").is_dir():
    raise FileNotFoundError(
        f"CIFAR-100 is missing from Google Drive: {DATA_ROOT / 'cifar-100-python'}"
    )
c100 = datasets.CIFAR100(CONFIG["data_root"], train=False, download=CONFIG["download_data"])
c100_names = c100.classes
near_ids = [index for index, label in enumerate(c100.targets) if c100_names[label] in NEAR]
far_ids = [index for index, label in enumerate(c100.targets) if c100_names[label] in FAR]

# Every score below is extracted from an evaluation-transform view. This prevents augmentation noise
# from changing score comparisons.
train_eval = CIFARWithTransform(raw_train, train_ids, eval_tf)
val_eval = CIFARWithTransform(raw_train, val_ids, eval_tf)
test_eval = CIFARWithTransform(raw_test, range(len(raw_test)), eval_tf)
near_eval = CIFARWithTransform(c100, near_ids, eval_tf)
far_eval = CIFARWithTransform(c100, far_ids, eval_tf)


def extract_outputs(model, dataset):
    """Cache penultimate features [N, 512], known logits [N, 10], and integer labels."""
    model.eval()
    feature_batches, logit_batches, label_batches = [], [], []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    for images, labels in loader:
        with torch.no_grad():
            features, logits = feature_logits(model, images.to(DEVICE))
        feature_batches.append(features.cpu())
        logit_batches.append(logits.cpu())
        label_batches.append(labels.cpu())
    return {
        "features": torch.cat(feature_batches),
        "logits": torch.cat(logit_batches),
        "labels": torch.cat(label_batches),
    }


def evaluate_score(known_validation, known_test, unknown, score_name):
    """Use a known-only 95th percentile threshold, then report ranking and operating-point metrics."""
    threshold = float(torch.quantile(known_validation, 0.95))
    labels = np.concatenate([np.zeros(len(known_test)), np.ones(len(unknown))])
    values = np.concatenate([known_test.numpy(), unknown.numpy()])
    return {
        "score": score_name,
        "auroc": roc_auc_score(labels, values),
        "threshold": threshold,
        "known_test_acceptance": float((known_test <= threshold).float().mean()),
        "unknown_rejection": float((unknown > threshold).float().mean()),
        "fpr_at_95tpr": float((unknown <= threshold).float().mean()),
    }


### Frozen Vanilla score comparisons

The following cell extracts known and unknown evaluation outputs, then computes the fixed post-hoc scores and Vanilla/GCSC comparison rows. CIFAR-100 is accessed only after training decisions are complete.

In [ ]:
# Vanilla scores: all four methods share exactly the same frozen Vanilla outputs.
vanilla_train = extract_outputs(vanilla, train_eval)
vanilla_val = extract_outputs(vanilla, val_eval)
vanilla_test = extract_outputs(vanilla, test_eval)
vanilla_near = extract_outputs(vanilla, near_eval)
vanilla_far = extract_outputs(vanilla, far_eval)
means, diagonal_variance = class_statistics(vanilla_train)
vanilla_caches = {
    "validation": vanilla_val,
    "test": vanilla_test,
    "near": vanilla_near,
    "far": vanilla_far,
}
vanilla_scores = {}
for dataset_name, cache in vanilla_caches.items():
    vanilla_scores[dataset_name] = unknownness(cache, means, diagonal_variance)

rows = []
for score_name in ["msp", "mls", "energy", "mahalanobis"]:
    near_row = evaluate_score(
        vanilla_scores["validation"][score_name],
        vanilla_scores["test"][score_name],
        vanilla_scores["near"][score_name],
        score_name,
    )
    far_row = evaluate_score(
        vanilla_scores["validation"][score_name],
        vanilla_scores["test"][score_name],
        vanilla_scores["far"][score_name],
        score_name,
    )
    all_unknown = torch.cat([vanilla_scores["near"][score_name], vanilla_scores["far"][score_name]])
    all_row = evaluate_score(
        vanilla_scores["validation"][score_name],
        vanilla_scores["test"][score_name],
        all_unknown,
        score_name,
    )
    rows.append({"model": "vanilla", "comparison": "near", **near_row})
    rows.append({"model": "vanilla", "comparison": "far", **far_row})
    rows.append({"model": "vanilla", "comparison": "all", **all_row})

# The trained-model comparison uses MLS for Vanilla and GCSC as required.
gcsc_val = extract_outputs(gcsc, val_eval)
gcsc_test = extract_outputs(gcsc, test_eval)
gcsc_near = extract_outputs(gcsc, near_eval)
gcsc_far = extract_outputs(gcsc, far_eval)
for model_name, caches in {
    "vanilla": {
        "validation": vanilla_val,
        "test": vanilla_test,
        "near": vanilla_near,
        "far": vanilla_far,
    },
    "gcsc": {"validation": gcsc_val, "test": gcsc_test, "near": gcsc_near, "far": gcsc_far},
}.items():
    score_cache = {}
    for dataset_name, cache in caches.items():
        score_cache[dataset_name] = unknownness(cache)["mls"]
    score_cache["all"] = torch.cat([score_cache["near"], score_cache["far"]])
    closed_set_accuracy = accuracy_score(
        caches["test"]["labels"], caches["test"]["logits"].argmax(1)
    )
    for group in ["near", "far", "all"]:
        rows.append(
            {
                "model": model_name,
                "comparison": group,
                "closed_set_accuracy": closed_set_accuracy,
                **evaluate_score(
                    score_cache["validation"], score_cache["test"], score_cache[group], "mls"
                ),
            }
        )


### PROSER final scores

This cell evaluates known logits and the placeholder score using the selected PROSER model; it adds the final comparison rows without changing any score definitions.

In [ ]:
# PROSER is evaluated with only its first ten logits for closed-set classification and MLS.
def extract_proser(model, dataset):
    model.eval()
    features, known_logits, full_logits, labels = [], [], [], []
    for images, batch_labels in DataLoader(dataset, batch_size=256, shuffle=False):
        with torch.no_grad():
            penultimate, logits = feature_logits(model.backbone, images.to(DEVICE))
        features.append(penultimate.cpu())
        known_logits.append(logits[:, :10].cpu())
        full_logits.append(logits.cpu())
        labels.append(batch_labels.cpu())
    return {
        "features": torch.cat(features),
        "logits": torch.cat(known_logits),
        "full_logits": torch.cat(full_logits),
        "labels": torch.cat(labels),
    }


proser_datasets = {
    "validation": val_eval,
    "test": test_eval,
    "near": near_eval,
    "far": far_eval,
}
proser_caches = {}
for dataset_name, dataset in proser_datasets.items():
    proser_caches[dataset_name] = extract_proser(proser, dataset)
proser_mls = {name: -cache["logits"].max(1).values for name, cache in proser_caches.items()}
# Placeholder unknownness compares the strongest dummy logit to the strongest known logit.
proser_placeholder = {}
for dataset_name, cache in proser_caches.items():
    strongest_dummy = cache["full_logits"][:, 10:].max(1).values
    strongest_known = cache["logits"].max(1).values
    proser_placeholder[dataset_name] = strongest_dummy - strongest_known
for score_name, scores in {"mls": proser_mls, "placeholder": proser_placeholder}.items():
    scores["all"] = torch.cat([scores["near"], scores["far"]])
    csa = accuracy_score(proser_caches["test"]["labels"], proser_caches["test"]["logits"].argmax(1))
    for group in ["near", "far", "all"]:
        rows.append(
            {
                "model": "proser",
                "comparison": group,
                "closed_set_accuracy": csa,
                **evaluate_score(scores["validation"], scores["test"], scores[group], score_name),
            }
        )


### Open-set result table and score distributions

This cell writes the final machine-readable OSR table and displays and saves the three-panel score-distribution figure.

In [ ]:
results = pd.DataFrame(rows)
results.to_csv(RESULTS / "osr_metrics.csv", index=False)
if CONFIG["print_metrics"]:
    display(results)

# Compact required score-distribution figure for frozen Vanilla scores.
if CONFIG["plot"]:
    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    for axis, score_name in zip(axes, ["msp", "mls", "mahalanobis"]):
        axis.hist(
            vanilla_scores["test"][score_name], bins=40, density=True, alpha=0.45, label="known"
        )
        axis.hist(
            vanilla_scores["near"][score_name], bins=40, density=True, alpha=0.45, label="near"
        )
        axis.hist(vanilla_scores["far"][score_name], bins=40, density=True, alpha=0.45, label="far")
        axis.set_title(score_name.upper())
        axis.set_xlabel("unknownness")
        axis.legend()
    figure.tight_layout()
    figure.savefig(REPO_RESULTS_DIR / "score_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()


### Accepted-unknown examples

The fixed Vanilla MLS threshold identifies example failures. This cell saves the failure table and displays and saves the example grid.

In [ ]:
# Save three incorrectly accepted examples from each group under the fixed Vanilla MLS threshold.
mls_threshold = float(torch.quantile(vanilla_scores["validation"]["mls"], 0.95))
failure_rows = []
for group_name, dataset, cache in [
    ("near", near_eval, vanilla_near),
    ("far", far_eval, vanilla_far),
]:
    accepted = torch.where(vanilla_scores[group_name]["mls"] <= mls_threshold)[0][:3]
    for position in accepted.tolist():
        _, cifar100_label = dataset[position]
        failure_rows.append(
            {
                "group": group_name,
                "cifar100_test_index": int(dataset.indices[position]),
                "unknown_class": c100_names[cifar100_label],
                "predicted_cifar10_class": C10[int(cache["logits"][position].argmax())],
                "mls_unknownness": float(vanilla_scores[group_name]["mls"][position]),
                "threshold": mls_threshold,
            }
        )
failure_frame = pd.DataFrame(failure_rows)
failure_frame.to_csv(RESULTS / "vanilla_mls_accepted_failures.csv", index=False)
if CONFIG["print_metrics"]:
    print("Accepted unknowns under the Vanilla MLS threshold")
    display(failure_frame.round(4))
if CONFIG["plot"]:
    # Plot the same saved failures, with their true and predicted classes and MLS value.
    figure, axes = plt.subplots(2, 3, figsize=(12, 7))
    for axis, failure in zip(axes.flat, failure_rows):
        image, _ = c100[failure["cifar100_test_index"]]
        axis.imshow(image)
        axis.set_title(
            f"{failure['group']}: {failure['unknown_class']} -> "
            f"{failure['predicted_cifar10_class']}\n"
            f"MLS {failure['mls_unknownness']:.2f} <= {failure['threshold']:.2f}",
            fontsize=9,
        )
        axis.axis("off")
    for axis in list(axes.flat)[len(failure_rows) :]:
        axis.axis("off")
    figure.tight_layout()
    figure.savefig(REPO_RESULTS_DIR / "accepted_unknown_failures.png", dpi=150, bbox_inches="tight")
    plt.show()
